#### Water Electrolyzer Levelized Cost of Hydrogen (LCOH) Simulation 

In [10]:
# Also need to consider Water electrolyzer degradation and health monitoring in another file.
# This is related to the work tha tI have currently done on accelerated stress testing (AST)

# What does this system involve: 
# calculating and tracking the lifestime average cost per kilogram of hydrogen produced
# This factors into the capital expedentures (CAPEX) - electrolyzer/plant construction - and operating expedentures (OPEX) - maintenance/labor
# And variable inputs like electricity and prices and facility utilization rates
# To track LCOH, I use this as an example: https://www.iea.org/data-and-statistics/data-tools/levelised-cost-of-hydrogen-maps

# Inputs: (weather, grid prices, load)

# Module: (degradation and health monitoring)

# Module: (LCOH simulation - 1MW electrolyzer over a 10 year project lifetime)
import numpy as np
import pandas as pd
import random as rand

# Parameters for system 
plant_capacity_mw = 1 # 1 MW system 
plant_capacity_gw = 1e12 # 1 GW system 

project_life_yrs = 10
discount_rate = 0.08

# Costs
capex_investment = 1500000 # 1/5M total investment
annual_opex = 45000 # $45k/yr for insurance, maintenance, land
replacement_cost = 400000 # cost to replace stack in 6 years

# Technical Baseline
initial_eff = 55 # 55 kWh needed for 1 kg of H2
annual_degradation_rate = 0.015 # efficiency drops by 1.5% we assume

# Monte Carlo simulation to generate 8760 hour inputs
# This is applicable for the year
np.random.seed(rand.randint(30,70))
hours = np.arange(8760)
time_span = 8760

# SImulate volatile power price profile (in pittsburgh it's 0.14 to 0.22 per kWh)
mu_log = 3.5 # mean of standard log prices (usually ranges from 3.5-4)
sigma_log = 0.5 # standard deviation of log prices (0.5 to 1.5)

electricity_prices = np.random.lognormal(mu_log, sigma_log, time_span)
#electricity_prices = np.random.normal(loc=35, scale=15, size=time_span)
electricity_prices = np.clip(electricity_prices, 10,100)

# Simulate the power input 
power_input_mw = np.random.uniform(low=0.2, high=1.0, size=time_span)

# Create a dataframe for a single operating year
hourly_profile = pd.DataFrame({
    'Hour':hours,
    'Power_Input_MW': power_input_mw,
    'Electricity_Price_USD_MWh':electricity_prices
})

# Calculate annual electricity costs for a single year
hourly_profile['Hourly_Cost_USD'] = hourly_profile['Power_Input_MW'] * hourly_profile['Electricity_Price_USD_MWh']
annual_electricity_cost_baseline = hourly_profile['Hourly_Cost_USD'].sum()
annual_energy_consumed_kwh = hourly_profile['Power_Input_MW'].sum() * 1000 # Convert MWh to kWh

print(f"Baseline Year Electricity Cost: ${annual_electricity_cost_baseline:,.2f}")
print(f"Baseline Year Energy Consumed: {annual_energy_consumed_kwh:,.2f} kWh\n")

Baseline Year Electricity Cost: $197,203.65
Baseline Year Energy Consumed: 5,290,399.73 kWh



In [ ]:
# Lifestyle flow simulation 
years = np.arange(0, project_life_yrs + 1)
df_lifecycle = pd.DataFrame(index=years)

# Initialize arrays for cash flows
capex_flow = np.zeros(project_life_yrs + 1)
opex_flow = np.zeros(project_life_yrs + 1)
h2_produced_kg = np.zeros(project_life_yrs + 1)

# Investment phase 
capex_flow[0] = capex_investment 
h2_produced_kg[0] = 0

# Initialize efficiency
current_efficiency = initial_eff

# Loop through operational years
for year in range(1, project_life_yrs + 1):
    # Apply degradation to the efficiency of the system 
    if year > 1:
        #for i in range(hours):
            current_efficiency *= (1 + annual_degradation_rate)
    
    # Calculate H2 produced this year
    annual_h2_output = annual_energy_consumed_kwh / current_efficiency
    h2_produced_kg[year] = annual_h2_output
    
    # Track operational costs
    opex_flow[year] = annual_opex + annual_electricity_cost_baseline
    
    # Mid-life stack replacement trigger 
    if year == 6:
        capex_flow[year] = replacement_cost 

# Populate lifecycle dataframe
df_lifecycle['CAPEX'] = capex_flow
df_lifecycle['OPEX'] = opex_flow
df_lifecycle['Total_Cost'] = df_lifecycle['CAPEX'] + df_lifecycle['OPEX']
df_lifecycle['H2_Output_kg'] = h2_produced_kg

# Discount Factor = 1 / (1 + r)^t
df_lifecycle['Discount_Factor'] = 1 / ((1 + discount_rate) ** df_lifecycle.index)

# Multiply columns by the discount factor
df_lifecycle['Discounted_Costs'] = df_lifecycle['Total_Cost'] * df_lifecycle['Discount_Factor']
df_lifecycle['Discounted_H2_kg'] = df_lifecycle['H2_Output_kg'] * df_lifecycle['Discount_Factor']

# Calculate final LCOH
sum_discounted_costs = df_lifecycle['Discounted_Costs'].sum()
sum_discounted_h2 = df_lifecycle['Discounted_H2_kg'].sum()
lcoh = sum_discounted_costs / sum_discounted_h2

# Display results
print("--- Project Lifecycle Cast Flow ---")
print(df_lifecycle[['Total_Cost', 'H2_Output_kg', 'Discounted_Costs', 'Discounted_H2_kg']].round(2))
print("------------------------------------")
print(f"Final Levelized Cost of Hydrogen: ${lcoh:.2f} / kg")

--- Project Lifecycle Cast Flow ---
    Total_Cost  H2_Output_kg  Discounted_Costs  Discounted_H2_kg
0   1500000.00          0.00        1500000.00              0.00
1    242203.65      96189.09         224262.64          89063.97
2    242203.65      94767.57         207650.59          81247.92
3    242203.65      93367.07         192269.06          74117.79
4    242203.65      91987.26         178026.91          67613.38
5    242203.65      90627.84         164839.73          61679.79
6    642203.65      89288.51         404697.23          56266.91
7    242203.65      87968.98         141323.50          51329.05
8    242203.65      86668.94         130855.09          46824.53
9    242203.65      85388.12         121162.12          42715.32
10   242203.65      84126.23         112187.15          38966.72
------------------------------------
Final Levelized Cost of Hydrogen: $5.54 / kg


In [ ]:
# I can use seaborn to plot discounted costs vs discounted h2/kg over time 